In [ ]:
# FASTQ File Handling and Quality Control with Biopython

In [ ]:
# Objective:

# This notebook demonstrates how to read, analyse and filter FASTQ sequencing files using Biopython.
# It covers sequence parsing, PHRED quality scores, read statistics and quality-based filtering commonly performed during next-generation sequencing (NGS) data preprocessing.

In [ ]:
# Topics covered:

# 1. Introduction to FASTQ
# 2. Reading FASTQ Files
# 3. Parsing Sequencing Reads
# 4. Read Statistics
# 5. PHRED Quality Scores
# 6. Read Quality Assessment
# 7. Dataset Quality Statistics
# 8. Quality Filtering
# 9. Writing Filtered FASTQ Files

In [ ]:
! pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.3 MB/s eta 0:00:00


In [ ]:
from Bio import SeqIO

In [ ]:
# What is a FASTQ File?

# FASTQ is the standard file format used to store sequencing reads generated by next-generation sequencing (NGS) technologies.

# Unlike FASTA, each sequence in a FASTQ file is accompanied by a PHRED quality score that estimates the confidence of each base call.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving practice.fastq to practice.fastq


In [ ]:
# Reading FASTQ Files:
# Biopython uses `SeqIO.parse()` to read FASTQ records.
# Each record contains the sequence, sequence identifier and associated PHRED quality scores.

# Read Statistics:
# Basic sequencing statistics such as the number of reads and read lengths provide an initial overview of sequencing data quality and dataset composition.

In [ ]:
for record in SeqIO.parse("practice.fastq", "fastq"):
  print("Record id:",record.id)
  print("Sequence", record.seq)
  print("Length of sequence:", len(record.seq))
  print("Quality score:", record.letter_annotations["phred_quality"])
  print("-" * 40)

Record id: Read_1
Sequence ATGCGTAGCTAG
Length of sequence: 12
Quality score: [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40]
----------------------------------------
Record id: Read_2
Sequence CGTATCGATGCA
Length of sequence: 12
Quality score: [39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39]
----------------------------------------
Record id: Read_3
Sequence TTAGCGATCGTA
Length of sequence: 12
Quality score: [37, 37, 37, 37, 37, 40, 40, 40, 40, 40, 39, 39]
----------------------------------------
Record id: Read_4
Sequence GCTAGCATGATC
Length of sequence: 12
Quality score: [40, 40, 40, 40, 39, 39, 39, 39, 37, 37, 37, 37]
----------------------------------------
Record id: Read_5
Sequence AACCGGTTAACC
Length of sequence: 12
Quality score: [41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41]
----------------------------------------
Record id: Read_6
Sequence GGCTATCGATGC
Length of sequence: 12
Quality score: [40, 40, 40, 40, 40, 40, 0, 0, 0, 0, 40, 40]
-------------------------------------

In [ ]:
count = 0
for record in SeqIO.parse("practice.fastq", "fastq"):
  count += 1

print("Total reads:", count)

Total reads: 8


In [ ]:
longest_sequence = ""
longest_id = ""
longest_length = 0

for record in SeqIO.parse("practice.fastq", "fastq"):
  seq = record.seq

  if len(seq) > longest_length:
    longest_sequence = seq
    longest_id = record.id
    longest_length = len(seq)

print("Longest read:", longest_sequence)
print("Longest id:", longest_id)
print("Length of read:", longest_length)

Longest read: ATGCGTAGCTAG
Longest id: Read_1
Length of read: 12


In [ ]:
shortest_seq = ""
shortest_id = ""
shortest_length = 100

for record in SeqIO.parse("practice.fastq", "fastq"):
  seq = record.seq

  if len(seq) < shortest_length:
    shortest_seq = seq
    shortest_id = record.id
    shortest_length = len(seq)

print("Shortest sequence:", shortest_seq)
print("Shortest id:", shortest_id)
print("Length of sequence:", shortest_length)

Shortest sequence: ATGCGTAGCTAG
Shortest id: Read_1
Length of sequence: 12


In [ ]:
# PHRED Quality Score:
# PHRED scores estimate the probability that a nucleotide has been incorrectly identified during sequencing.
# Higher PHRED scores indicate greater confidence in the accuracy of the base call.
# As the PHRED score increases; probability of error drops significantly
# ! means 0 and is a poor base

# Read Quality Assessment:
# Average PHRED scores are commonly used to evaluate the quality of sequencing reads. Reads with low average quality are often removed before downstream analyses such as alignment or variant calling.

In [ ]:
for record in SeqIO.parse("practice.fastq", "fastq"):
  print("Sequence:", record.seq)
  quality_score = record.letter_annotations["phred_quality"]
  print("Quality scores:", quality_score)
  print("Maximum score:", max(quality_score))
  print("Minimum score:", min(quality_score))
  print("Average score:", round(sum(quality_score)/len(quality_score),2))
  print("-" * 40)

Sequence: ATGCGTAGCTAG
Quality scores: [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40]
Maximum score: 40
Minimum score: 40
Average score: 40.0
----------------------------------------
Sequence: CGTATCGATGCA
Quality scores: [39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39]
Maximum score: 39
Minimum score: 39
Average score: 39.0
----------------------------------------
Sequence: TTAGCGATCGTA
Quality scores: [37, 37, 37, 37, 37, 40, 40, 40, 40, 40, 39, 39]
Maximum score: 40
Minimum score: 37
Average score: 38.58
----------------------------------------
Sequence: GCTAGCATGATC
Quality scores: [40, 40, 40, 40, 39, 39, 39, 39, 37, 37, 37, 37]
Maximum score: 40
Minimum score: 37
Average score: 38.67
----------------------------------------
Sequence: AACCGGTTAACC
Quality scores: [41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41]
Maximum score: 41
Minimum score: 41
Average score: 41.0
----------------------------------------
Sequence: GGCTATCGATGC
Quality scores: [40, 40, 40, 40, 40, 40, 0, 0, 0, 

In [ ]:
# FASTQ Quality statistics

In [ ]:
# Highest average read

highest_average = 0
highest_id = ""
highest_seq = ""
for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality)/ len(quality)

  if average > highest_average:
    highest_average = average
    highest_id = record.id
    highest_seq = record.seq

print("Highest Quality READ:", highest_seq)
print("READ id:", highest_id)
print("Average PHRED", highest_average)

Highest Quality READ: AACCGGTTAACC
READ id: Read_5
Average PHRED 41.0


In [ ]:
# Lowest Average Quality Read

lowest_seq = ""
lowest_average = float("inf")
lowest_id = ""

for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality) / len(quality)

  if average < lowest_average:
    lowest_seq = record.seq
    lowest_average = average
    lowest_id = record.id

print("Lowest Quality READ:", lowest_seq)
print("READ id:", lowest_id)
print("Average PHRED:",(round(lowest_average,2)))

Lowest Quality READ: GGCTATCGATGC
READ id: Read_6
Average PHRED: 26.67


In [ ]:
# Average quality of the Read

total_average = 0
number_of_reads = 0

for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality)/len(quality)

  total_average += average
  number_of_reads += 1

dataset_average = total_average / number_of_reads
print("Dataset average:", round(dataset_average,2))

Dataset average: 37.71


In [ ]:
# Calculating high quality Read

high_quality_reads = 0

for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality) / len(quality)

  if average >= 30:
    high_quality_reads += 1

print("High-quality reads:", high_quality_reads)


High-quality reads: 7
Low quality reads: 11


In [ ]:
# Calculating low quality read

low_quality_reads = 0

for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality) / len(quality)

  if average < 32:
    low_quality_reads += 1

print("Low quality reads:", low_quality_reads)

Low quality reads: 1


In [ ]:
# Quality Filtering

# Quality filtering removes unreliable sequencing reads from the dataset, improving the accuracy of downstream bioinformatics analyses.

In [ ]:
# Writing FASTQ file

high_quality_reads = []
total_reads = 0

for record in SeqIO.parse("practice.fastq", "fastq"):
  total_reads += 1

  quality = record.letter_annotations["phred_quality"]
  average = sum(quality) / len(quality)

  if average >= 30:
    high_quality_reads.append(record)

print("Total reads before filtering:", total_reads)
print("Total number of high quality reads:", len(high_quality_reads))
written = SeqIO.write(high_quality_reads, "filtered.fastq", "fastq")
print("Reads written:", written)

Total reads before filtering: 8
Total number of high quality reads: 7
Reads written: 7


In [ ]:
low_quality_reads = []

for record in SeqIO.parse("practice.fastq", "fastq"):
  quality = record.letter_annotations["phred_quality"]
  average = sum(quality)/len(quality)

  if average < 30:
    low_quality_reads.append(record)

print("Total number of low quality reads:", len(low_quality_reads))

for record in low_quality_reads:
  print(record.id)

Total number of low quality reads: 1
Read_6


In [ ]:
# Workflow:
#         Raw FASTQ
#             ↓
#         Read FASTQ
#             ↓
#         Extract Sequences & PHRED Scores
#             ↓
#         Calculate Read Statistics
#             ↓
#         Assess Read Quality
#             ↓
#         Filter Low-Quality Reads
#             ↓
#         Export High-Quality FASTQ

In [ ]:
# Summary

# In this notebook, I have explored FASTQ file handling and sequencing quality assessment using Biopython.

# By the end of this notebook, I am able to:
# - Read FASTQ files using SeqIO
# - Parse sequencing reads
# - Access PHRED quality scores
# - Calculate read statistics
# - Assess sequencing quality
# - Identify high-quality and low-quality reads
# - Filter reads based on quality thresholds
# - Export filtered FASTQ files
